In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from xgboost import XGBClassifier

In [61]:
df= pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn (1).csv')

In [62]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [63]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [64]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


# data cleaning

In [65]:
print(df.isnull().sum().sum()) 

0


In [66]:
df.drop('customerID', axis=1, inplace=True)

In [67]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['tenure'] = df['tenure'].replace(0, 1)



In [68]:
# 3) Handle Missing Values

df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [69]:
# 4) Convert Target
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})


In [70]:
#convert feature 
binary_cols = [
    'Partner','Dependents','PhoneService',
    'PaperlessBilling'
]

for col in binary_cols:
    df[col] = df[col].map({'Yes':1, 'No':0})

In [71]:
# 6) Fix tricky columns
cols_to_fix = [
    'MultipleLines','OnlineSecurity','OnlineBackup',
    'DeviceProtection','TechSupport',
    'StreamingTV','StreamingMovies'
]

for col in cols_to_fix:
    df[col] = df[col].replace({
        'No internet service': 'No',
        'No phone service': 'No'
    })

In [72]:
# 7) Feature Engineering 

df['AvgCharges'] = df['TotalCharges'] / df['tenure']

df['ExpectedTotal'] = df['MonthlyCharges'] * df['tenure']
df['Charge_Error'] = df['TotalCharges'] - df['ExpectedTotal']

df['IsNew'] = (df['tenure'] < 12).astype(int)
df['IsLongContract'] = (df['Contract'] == 'Two year').astype(int)

In [73]:
df.replace([np.inf, -np.inf], 0, inplace=True)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgCharges,ExpectedTotal,Charge_Error,IsNew,IsLongContract
0,Female,0,1,0,1,0,No,DSL,No,Yes,...,1,Electronic check,29.85,29.85,0,29.850000,29.85,0.00,1,0
1,Male,0,0,0,34,1,No,DSL,Yes,No,...,0,Mailed check,56.95,1889.50,0,55.573529,1936.30,-46.80,0,0
2,Male,0,0,0,2,1,No,DSL,Yes,Yes,...,1,Mailed check,53.85,108.15,1,54.075000,107.70,0.45,1,0
3,Male,0,0,0,45,0,No,DSL,Yes,No,...,0,Bank transfer (automatic),42.30,1840.75,0,40.905556,1903.50,-62.75,0,0
4,Female,0,0,0,2,1,No,Fiber optic,No,No,...,1,Electronic check,70.70,151.65,1,75.825000,141.40,10.25,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,1,1,24,1,Yes,DSL,Yes,No,...,1,Mailed check,84.80,1990.50,0,82.937500,2035.20,-44.70,0,0
7039,Female,0,1,1,72,1,Yes,Fiber optic,No,Yes,...,1,Credit card (automatic),103.20,7362.90,0,102.262500,7430.40,-67.50,0,0
7040,Female,0,1,1,11,0,No,DSL,Yes,No,...,1,Electronic check,29.60,346.45,0,31.495455,325.60,20.85,1,0
7041,Male,1,1,0,4,1,Yes,Fiber optic,No,No,...,1,Mailed check,74.40,306.60,1,76.650000,297.60,9.00,1,0


In [74]:
# 8) Encoding
df = pd.get_dummies(df, drop_first=True)

In [75]:
df = df.astype(int)

In [76]:
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,AvgCharges,...,OnlineBackup_Yes,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,0,1,0,1,29,29,0,29,...,1,0,0,0,0,0,0,0,1,0
1,0,0,0,34,1,0,56,1889,0,55,...,0,1,0,0,0,1,0,0,0,1
2,0,0,0,2,1,1,53,108,1,54,...,1,0,0,0,0,0,0,0,0,1
3,0,0,0,45,0,0,42,1840,0,40,...,0,1,1,0,0,1,0,0,0,0
4,0,0,0,2,1,1,70,151,1,75,...,0,0,0,0,0,0,0,0,1,0


In [77]:
print(df.isnull().sum())

SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
PaperlessBilling                         0
MonthlyCharges                           0
TotalCharges                             0
Churn                                    0
AvgCharges                               0
ExpectedTotal                            0
Charge_Error                             0
IsNew                                    0
IsLongContract                           0
gender_Male                              0
MultipleLines_Yes                        0
InternetService_Fiber optic              0
InternetService_No                       0
OnlineSecurity_Yes                       0
OnlineBackup_Yes                         0
DeviceProtection_Yes                     0
TechSupport_Yes                          0
StreamingTV_Yes                          0
StreamingMo

In [78]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [79]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))

Accuracy: 0.8019872249822569


In [81]:
# 13) Feature Importance

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(10))

Contract_Two year                 0.223314
InternetService_Fiber optic       0.173109
IsLongContract                    0.154076
Contract_One year                 0.081274
InternetService_No                0.066933
PaymentMethod_Electronic check    0.029881
IsNew                             0.025740
tenure                            0.025207
StreamingMovies_Yes               0.023647
OnlineSecurity_Yes                0.017622
dtype: float32
